# Paired Edit Validation

Run paired-edit checkpoint validation in Colab and generate `input / output / target` comparison sheets.


## What This Notebook Does

- Mounts Google Drive
- Clones `nail-retouch-assistant`
- Clones upstream `img2img-turbo`
- Installs validation dependencies
- Finds the latest `model_*.pkl` checkpoint in Drive unless you override it
- Runs validation on the 5 held-out test pairs
- Applies the pix2pix checkpoint compatibility patch before inference
- Writes comparison sheets to `/content/drive/MyDrive/nail-retouch-paired-validation/<checkpoint_name>/`
- Mirrors the same files to `/content/workdir/paired_edit/validation/<checkpoint_name>/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_ROOT = Path('/content/nail-retouch-assistant')
UPSTREAM_ROOT = Path('/content/img2img-turbo')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(
    ['git', 'clone', 'https://github.com/Zifpen/nail-retouch-assistant.git', str(PROJECT_ROOT)],
    check=True,
)

if UPSTREAM_ROOT.exists():
    shutil.rmtree(UPSTREAM_ROOT)
subprocess.run(
    ['git', 'clone', 'https://github.com/GaParmar/img2img-turbo.git', str(UPSTREAM_ROOT)],
    check=True,
)

print('repo head:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD']).decode().strip())


In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'pyyaml', 'diffusers', 'transformers', 'accelerate', 'peft', 'sentencepiece'], check=True)


In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = PROJECT_ROOT / 'colab' / 'paired_edit_phase1_oldroute_config.yaml'
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

DATASET_DIR = Path(cfg['drive_dataset_dir'])
CHECKPOINT_DIR = Path(cfg['drive_output_dir']) / 'checkpoints'
VALIDATION_OUT_ROOT = Path('/content/drive/MyDrive/nail-retouch-paired-core-v1-validation')
LOCAL_VALIDATION_OUT_ROOT = Path('/content/workdir/paired_edit/validation')
PAIRED_PROMPT = cfg['paired_prompt']

# Optional override: set to a specific checkpoint file, or leave as None to auto-pick latest.
CHECKPOINT_OVERRIDE = None

assert DATASET_DIR.exists(), f'Missing dataset dir: {DATASET_DIR}'
assert CHECKPOINT_DIR.exists(), f'Missing checkpoint dir: {CHECKPOINT_DIR}'

model_candidates = sorted(
    CHECKPOINT_DIR.glob('model_*.pkl'),
    key=lambda p: int(p.stem.split('_')[-1]),
)
assert model_candidates, f'No model_*.pkl found in {CHECKPOINT_DIR}'

CHECKPOINT_PATH = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else model_candidates[-1]
print('dataset:', DATASET_DIR)
print('checkpoint:', CHECKPOINT_PATH)
print('validation output root:', VALIDATION_OUT_ROOT)
print('local mirror root:', LOCAL_VALIDATION_OUT_ROOT)
print('prompt:', PAIRED_PROMPT)


In [ ]:
cmd = [
    'python',
    str(PROJECT_ROOT / 'src' / 'paired_edit' / 'run_local_validation.py'),
    '--checkpoint',
    str(CHECKPOINT_PATH),
    '--dataset-dir',
    str(DATASET_DIR),
    '--upstream-dir',
    str(UPSTREAM_ROOT),
    '--output-root',
    str(VALIDATION_OUT_ROOT),
    '--mirror-output-root',
    str(LOCAL_VALIDATION_OUT_ROOT),
    '--prompt',
    PAIRED_PROMPT,
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path

out_dir = Path('/content/drive/MyDrive/nail-retouch-paired-core-v1-validation') / CHECKPOINT_PATH.stem
print('validation output dir:', out_dir)
for path in sorted(out_dir.glob('*')):
    print(path)
